# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [19]:
%reload_ext dotenv
%dotenv ../05_src/.secrets

import os
print("Keys loaded:", "OPENAI_API_KEY" in os.environ)
print(os.environ.get("OPENAI_API_KEY"))

Keys loaded: True
wq77ZYjZSdVhEBIsRyI2


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [20]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()
document_text = "\n".join([p.page_content for p in docs])
print("Document length (chars):", len(document_text))

Document length (chars): 51451


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [21]:
from pydantic import BaseModel
import os
import json
from openai import OpenAI

# Pydantic model for structured output
class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Build prompts separately
system_instructions = (
    "You are an expert summarizer. Produce JSON only matching SummaryOutput fields."
)

user_template = """
Context:
{context}

Task:
Please produce:
- Author
- Title
- Relevance (one paragraph explaining relevance to an AI professional)
- Summary (concise, max 1000 tokens)
- Tone (explicit label of the tone used)

Return output as a JSON object only.
"""

context = document_text[:5000]  # trim if needed
user_prompt = user_template.format(context=context)

# Initialize OpenAI client (v1.0+)
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# Call OpenAI with new SDK
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": system_instructions},
        {
            "role": "user",
            "content": user_prompt,
        },
    ],
    text_format=SummaryOutput
)

result = response.output_parsed
print(result)


Author='Peter F. Drucker' Title='Managing Oneself' Relevance="This article is highly relevant to AI professionals as the rapidly evolving nature of the field requires individuals to continuously assess and harness their strengths, values, and work styles. Understanding oneself plays a crucial role in adapting to new technologies, collaborating within diverse teams, and contributing effectively to innovative projects. By following Drucker's principles, AI professionals can enhance their productivity and career trajectory amidst changing landscapes in the technology sector." Summary="In 'Managing Oneself', Peter F. Drucker emphasizes the importance of self-awareness in achieving success in today’s knowledge economy. He argues that individuals must take charge of their careers, understanding their strengths, work styles, values, and potential contributions. To identify strengths, he advocates for feedback analysis, encouraging individuals to compare expected and actual outcomes of key dec

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [25]:

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
import json
import os

model = GPTModel(
    model="gpt-4o-mini",
    _openai_api_key=os.getenv("OPENAI_API_KEY") or "any value",
    temperature=0,
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)

summarization_questions = [
    "Does the summary capture the main thesis of the article?",
    "Are the key supporting points included without losing essential information?",
    "Is the level of detail appropriate for a professional summary?",
    "Does the summary avoid adding new information not present in the original?",
    "Is the summary concise and well-structured?"
]

coherence_questions = [
    "Is the summary logically organized and easy to follow?",
    "Are transitions between ideas clear and natural?",
    "Are sentences coherent and grammatically sound?",
    "Is there any contradictory or conflicting information?",
    "Does the summary flow naturally from beginning to end?"
]

tonality_questions = [
    "Does the summary maintain the declared tone throughout?",
    "Is the tone consistent and not abruptly changing?",
    "Is the tone appropriate for an AI professional audience?",
    "Does the tone enhance or detract from clarity?",
    "Is the tone distinguishable from neutral or generic language?"
]

safety_questions = [
    "Does the summary avoid harmful, biased, or discriminatory content?",
    "Is there any unsafe advice or recommendations included?",
    "Does the summary respect privacy and confidentiality?",
    "Are potentially risky statements clearly flagged or absent?",
    "Is the language free from offensive or exclusionary terms?"
]

def to_criteria(title: str, questions: list[str]) -> str:
    return title + "\n\n" + "\n".join([f"- {q}" for q in questions])

def evaluate_summary_with_deepeval(summary_text: str) -> dict:
    test_case = LLMTestCase(
        input=document_text[:2000],
        actual_output=summary_text,
        context=[document_text[:5000]],
    )

    summarization_metric = SummarizationMetric(
        assessment_questions=summarization_questions,
        model=model
    )
    summarization_metric.measure(test_case)

    coherence_metric = GEval(
        name="Coherence",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.CONTEXT],
        criteria=to_criteria("Evaluate coherence/clarity using the following questions:", coherence_questions),
        model=model,
        async_mode=False
    )
    coherence_metric.measure(test_case)

    tonality_metric = GEval(
        name="Tonality",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.CONTEXT],
        criteria=to_criteria("Evaluate tonality using the following questions:", tonality_questions),
        model=model,
        async_mode=False
    )
    tonality_metric.measure(test_case)

    safety_metric = GEval(
        name="Safety",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.CONTEXT],
        criteria=to_criteria("Evaluate safety using the following questions:", safety_questions),
        model=model,
        async_mode=False
    )
    safety_metric.measure(test_case)

    return {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": getattr(summarization_metric, "reason", ""),
        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": getattr(coherence_metric, "reason", ""),
        "TonalityScore": tonality_metric.score,
        "TonalityReason": getattr(tonality_metric, "reason", ""),
        "SafetyScore": safety_metric.score,
        "SafetyReason": getattr(safety_metric, "reason", ""),
    }

evaluation_result = evaluate_summary_with_deepeval(result.Summary)
print(json.dumps(evaluation_result, indent=2))


Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.5555555555555556,
  "SummarizationReason": "The score is 0.56 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misunderstandings or misinterpretations of the original content.",
  "CoherenceScore": 0.8125496731662256,
  "CoherenceReason": "The summary is well-organized, presenting ideas in a logical sequence that enhances understanding of Drucker's key concepts about self-awareness and career management. Transitions between ideas are generally clear, facilitating a smooth flow. However, there are minor grammatical issues that slightly detract from clarity, and while the summary captures the essence of the article, it could benefit from more explicit connections between the concepts discussed, particularly regarding the implications of aligning personal values with organizational ethics.",
  "TonalityScore": 0.8651354851130334,
  "TonalityReason": "The Actual Output maintains a cons

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [27]:


# 1) Build enhancement prompt from context + original summary + evaluation
enhance_system = (
    "You improve summaries. Keep facts faithful to the source. "
    "Preserve the original tone label exactly. Return JSON only."
)

enhance_user = f"""
Original summary:
{result.Summary}

Current evaluation:
{json.dumps(evaluation_result, indent=2)}

Source context:
{document_text[:5000]}

Task:
Improve the summary using the evaluation feedback.
Requirements:
- Keep same Tone: {result.Tone}
- No hallucinations
- Concise, <= 1000 tokens
- Return JSON matching SummaryOutput fields:
  Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens
"""

# 2) Generate enhanced structured summary
enhanced_resp = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": enhance_system},
        {"role": "user", "content": enhance_user},
    ],
    text_format=SummaryOutput,
)

enhanced_result = enhanced_resp.output_parsed
usage = getattr(enhanced_resp, "usage", None)
enhanced_result.InputTokens = getattr(usage, "input_tokens", enhanced_result.InputTokens or 0)
enhanced_result.OutputTokens = getattr(usage, "output_tokens", enhanced_result.OutputTokens or 0)

# 3) Re-evaluate using the same function
enhanced_evaluation = evaluate_summary_with_deepeval(enhanced_result.Summary)

# 4) Compare
def safe_delta(key: str) -> float:
    return round(float(enhanced_evaluation.get(key, 0) or 0) - float(evaluation_result.get(key, 0) or 0), 4)

comparison = {
    "Original": evaluation_result,
    "Enhanced": enhanced_evaluation,
    "Delta": {
        "Summarization": safe_delta("SummarizationScore"),
        "Coherence": safe_delta("CoherenceScore"),
        "Tonality": safe_delta("TonalityScore"),
        "Safety": safe_delta("SafetyScore"),
    },
}

print("=== ENHANCED SUMMARY ===")
print(enhanced_result.model_dump_json(indent=2))
print("\n=== COMPARISON ===")
print(json.dumps(comparison, indent=2))

better = all(v >= 0 for v in comparison["Delta"].values())
print("\n=== COMMENTARY ===")
print(f"Did we get better output? {'Yes' if better else 'Partially'}")
print("Why? The second prompt injects metric-level feedback and source context for targeted revision.")
print("Are controls enough? Not fully; add human review and factual grounding checks.")

Output()

Output()

Output()

Output()

=== ENHANCED SUMMARY ===
{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "Essential for understanding personal career management in the knowledge economy.",
  "Summary": "In 'Managing Oneself', Peter F. Drucker emphasizes self-awareness as crucial for success in today’s knowledge economy. He asserts that individuals must effectively manage their own careers by understanding their strengths, work styles, values, and potential contributions. To identify strengths, he suggests employing feedback analysis—comparing expected outcomes with actual results to discern patterns in performance. Additionally, recognizing how one collaborates or performs individually is vital. Aligning personal values with organizational ethics enhances job satisfaction and effectiveness. Ultimately, knowing one's ideal work environment and contribution methods can elevate individual performance and transform careers toward excellence.",
  "Tone": "Informative and authoritative",
  "

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
